# XGBoost

Standard tabular ML workflow for LD50 regression.


In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np

PROJECT_ROOT = Path("../..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.modeling import artifact_paths, load_tabular_arrays, save_ml_run

In [2]:
MODEL_NAME = "xgboost"
BASE_DIR = Path.cwd()
RANDOM_STATE = 42

X_train, X_valid, X_test, y_train, y_valid, y_test, feature_names = load_tabular_arrays('../../data/feature_selection')
X_train = X_train.astype(np.float32)
X_valid = X_valid.astype(np.float32)
X_test = X_test.astype(np.float32)

paths = artifact_paths(
    BASE_DIR,
    MODEL_NAME,
    model_suffix=".pkl",
    include_importance=True,
)


In [3]:
# Hypertuning
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

xgb_device = "cuda"

search = RandomizedSearchCV(
    estimator=XGBRegressor(
        tree_method="hist",
        device=xgb_device,
        random_state=RANDOM_STATE,
    ),
    param_distributions={
        "n_estimators": [700, 900, 1000, 1200],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "max_depth": [3, 4, 6, 8],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "reg_lambda": [0.5, 1.0, 2.0, 5.0],
    },
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=False,
)

try:
    search.fit(X_valid, y_valid)
except Exception as exc:
    warnings.warn(f"XGBoost CUDA hypertuning failed; retrying on CPU. Details: {exc}")
    xgb_device = "cpu"
    search.estimator.set_params(device=xgb_device)
    search.fit(X_valid, y_valid)

best_params = search.best_params_
best_device = xgb_device
print(f"[XGBoost hypertuning] Best params: {best_params} | validation RMSE: {-search.best_score_:.4f}")


[XGBoost hypertuning] Best params: {'subsample': 0.7, 'reg_lambda': 1.0, 'n_estimators': 900, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 1.0} | validation RMSE: 0.8141


In [4]:
model = XGBRegressor(
    **best_params,
    tree_method="hist",
    device=best_device,
    random_state=RANDOM_STATE,
)

try:
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
except Exception as exc:
    warnings.warn(f"XGBoost CUDA failed; retrying on CPU. Details: {exc}")
    model.set_params(device="cpu")
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

predictions = model.predict(X_test)


c:\Users\Tommaso\miniconda3\envs\tox_prediction\lib\site-packages\xgboost\core.py:751: UserWarning: [16:28:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [5]:
metrics = save_ml_run("XGBoost", model, paths, X_test, y_test, predictions, feature_names)

[XGBoost] RMSE: 0.9260 | MAE: 0.6659 | R2: 0.2347
